# Importing libraries

In [ ]:
from datasets import load_dataset
import nlpaug.augmenter.word as naw
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
import torch
import gc
from tqdm import tqdm
import re

tqdm.pandas()


rseed = 42

# Importing HuggingFace Token

In [10]:
from huggingface_hub import login
from getpass import getpass

hftoken = getpass("HF-Token ")
login(token=hftoken)

In [ ]:
#reading finished datasets locally, for further processing and metrics, so the augmentation process is not needed again
syn_df = pd.read_csv("../data/mbti_augmented_agg/syn_augmented_agg.csv", sep = "\t", quoting = 1)
bt_df = pd.read_csv("../data/mbti_augmented_agg/bt_augmented_agg.csv", sep = "\t", quoting = 1)
sw_df = pd.read_csv("../data/mbti_augmented_agg/sw_augmented_agg.csv", sep = "\t", quoting = 1)
del_df = pd.read_csv("../data/mbti_augmented_agg/del_augmented_agg.csv", sep = "\t", quoting = 1)

bt_df.head()

,post,labels,I,N,F,P,post_augmented
0,"mechanic's okay, never had problems with it, a...",15,1,0,0,1,"Mechanics is okay, never had any problems with..."
1,"truee,, hell yeah, we pass the test without us...",15,1,0,0,1,"truee, damn it, yes, we pass the test without ..."
2,missed that. it doesn't sound like your typica...,15,1,0,0,1,I missed that. it doesn't sound like your typi...
3,probably because you have a nice life. there's...,15,1,0,0,1,"Probably because you have a good life, there's..."
4,quite a bit of differences. in this general bi...,15,1,0,0,1,quite a little bit of differences. in this gen...


In [ ]:
sw_df.info()
del_df.info()
bt_df.info()
syn_df.info()


## Evaluating MBTI label masking


In [ ]:
print(f"bt_df mbti mask counts in post col (pre aug): {bt_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"bt_df mbti mask counts in augmented col: {bt_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")
#print(f"bt_df failed mbti masking example, counts in augmented col: {bt_df["post_augmented"].apply(lambda x: "MBTITOPOKENPLACECCHOLDER" in x).sum()}")
print(f"bt_df john mask counts in post col (pre aug): {bt_df["post"].apply(lambda x: "john" in x).sum()}")

print(f"syn_df mbti mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"syn_df mbti mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"syn_df UNK mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "UNK" in x).sum()}")
print(f"syn_df john mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "john" in x).sum()}")


print(f"sw_df mbti mask counts in post col (pre aug): {sw_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"sw_df mbti mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"sw_df mbti mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

print(f"del_df mbti mask counts in post col (pre aug): {del_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"del_df mbti mask counts in augmented col: {del_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

In [2]:
mask_lost = (syn_df["post"].str.contains("<mbti>") & ~syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with failed masking: {mask_lost.sum()}")

mask_lost_2 = (~syn_df["post"].str.contains("<mbti>") & syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with failed masking: {mask_lost_2.sum()}")

mask_ok = (syn_df["post"].str.contains("<mbti>") & syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with successful masking: {mask_ok.sum()}")

total_mask_lost = mask_lost | mask_lost_2
syn_df_final = syn_df[~total_mask_lost].reset_index(drop=True)



Rows with failed masking: 0
Rows with failed masking: 128
Rows with successful masking: 1601


In [3]:
mask_lost = (bt_df["post"].str.contains("<mbti>") & ~bt_df["post_augmented"].str.contains("<mbti>"))


failed_masking = pd.DataFrame(bt_df[mask_lost])
print(f"Rows with failed masking: {mask_lost.sum()}")

mask_ok = (bt_df["post"].str.contains("<mbti>") & bt_df["post_augmented"].str.contains("<mbti>"))
bt_df_final = bt_df[~mask_lost].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

Rows with failed masking: 29
Rows with successful masking: 5544


In [4]:
mask_lost_sw = (sw_df["post"].str.contains("<mbti>") & ~sw_df["post_augmented"].str.contains("<mbti>"))
print(f"Rows with failed masking: {mask_lost_sw.sum()}")

mask_ok = (sw_df["post"].str.contains("<mbti>") & sw_df["post_augmented"].str.contains("<mbti>"))
sw_df_final = sw_df[~mask_lost_sw].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

mask_lost_del = (del_df["post"].str.contains("<mbti>") & ~del_df["post_augmented"].str.contains("<mbti>"))
print(f"Rows with failed masking: {mask_lost_del.sum()}")

mask_ok = (del_df["post"].str.contains("<mbti>") & del_df["post_augmented"].str.contains("<mbti>"))
del_df_final = del_df[~mask_lost_del].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

Rows with failed masking: 0
Rows with successful masking: 389
Rows with failed masking: 0
Rows with successful masking: 392


In [ ]:
print(f"bt_df </s> mask counts in post col (pre aug): {bt_df["post"].apply(lambda x: "</s>" in x).sum()}")
print(f"bt_df </s> mask counts in augmented col: {bt_df["post_augmented"].apply(lambda x: "</s>" in x).sum()}")
#print(f"bt_df failed mbti masking example, counts in augmented col: {bt_df["post_augmented"].apply(lambda x: "MBTITOPOKENPLACECCHOLDER" in x).sum()}")


print(f"syn_df </s> mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "</s>" in x).sum()}")
print(f"syn_df </s> mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "</s>>" in x).sum()}")
print(f"syn_df UNK mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "UNK" in x).sum()}")
print(f"syn_df john mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "john" in x).sum()}")


print(f"sw_df </s> mask counts in post col (pre aug): {sw_df["post"].apply(lambda x: "</s>" in x).sum()}")
print(f"sw_df </s> mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "</s>" in x).sum()}")

print(f"del_df </s> mask counts in post col (pre aug): {del_df["post"].apply(lambda x: "</s>" in x).sum()}")
print(f"del_df </s> mask counts in augmented col: {del_df["post_augmented"].apply(lambda x: "</s>" in x).sum()}")

In [ ]:
mask_lost = (bt_df["post"].str.contains("</s>") & ~bt_df["post_augmented"].str.contains("</s>"))


failed_masking = pd.DataFrame(bt_df[mask_lost])
print(f"Rows with failed masking: {mask_lost.sum()}")

mask_ok = (bt_df["post"].str.contains("</s>") & bt_df["post_augmented"].str.contains("</s>"))
#bt_df_final = bt_df[~mask_lost].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

mask_lost = (syn_df["post"].str.contains("</s>") & ~syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with failed masking: {mask_lost.sum()}")

mask_lost_2 = (~syn_df["post"].str.contains("</s>") & syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with failed masking: {mask_lost_2.sum()}")

mask_ok = (syn_df["post"].str.contains("</s>") & syn_df["post_augmented"].str.contains("[UNK]"))
print(f"Rows with successful masking: {mask_ok.sum()}")

# total_mask_lost = mask_lost | mask_lost_2
# syn_df_final = syn_df[~total_mask_lost].reset_index(drop=True)


In [ ]:
mask_lost_sw = (sw_df["post"].str.contains("</s>") & ~sw_df["post_augmented"].str.contains("</s>"))
print(f"Rows with failed masking: {mask_lost_sw.sum()}")

mask_ok = (sw_df["post"].str.contains("</s>") & sw_df["post_augmented"].str.contains("</s>"))
sw_df_final = sw_df[~mask_lost_sw].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

mask_lost_del = (del_df["post"].str.contains("</s>") & ~del_df["post_augmented"].str.contains("</s>"))
print(f"Rows with failed masking: {mask_lost_del.sum()}")

mask_ok = (del_df["post"].str.contains("</s>") & del_df["post_augmented"].str.contains("</s>"))
del_df_final = del_df[~mask_lost_del].reset_index(drop=True)
print(f"Rows with successful masking: {mask_ok.sum()}")

## Merging the samples together
Since I divided per label I need to put the samples together again and then merge them to the initial dataset.

In [5]:
# putting them all together
#bt_df = bt_df.drop(columns = "post_augmented")
bt_df_final = bt_df_final.drop(columns = "post")
bt_df_final = bt_df_final.rename(columns={"post_augmented": "post"})

syn_df_final = syn_df_final.drop(columns = "post")
syn_df_final = syn_df_final.rename(columns={"post_augmented": "post"})

sw_df_final = sw_df_final.drop(columns = "post")
sw_df_final = sw_df_final.rename(columns={"post_augmented": "post"})

del_df_final = del_df_final.drop(columns = "post")
del_df_final = del_df_final.rename(columns={"post_augmented": "post"})

In [6]:
df_augmented_final = pd.concat([syn_df_final, sw_df_final, del_df_final, bt_df_final])

In [12]:
df = pd.concat([df, df_augmented_final])


In [13]:
df["labels"].value_counts().reset_index()

,labels,count
0,9,9182
1,8,7570
2,11,6372
3,10,5338
4,1,3429
5,3,3383
6,15,1995
7,13,1989
8,12,1988
9,7,1988


# Undersampling
To bring all label counts approximately on the same level, label above the chosen threshold must be undersampled. This is simply done by randomly deleting observations.

In [14]:
from sklearn.utils import resample

def undersample(df, target_col, target_n):
    sampled = []
    for cls in df[target_col].unique():
        cls_df = df[df[target_col] == cls]
        if len(cls_df) > target_n:
            cls_df = resample(cls_df, n_samples=target_n, 
                            random_state=42, replace=False)
        sampled.append(cls_df)
    return pd.concat(sampled).reset_index(drop=True)

df_undersampled = undersample(df, "labels", 2000)

In [15]:
df_undersampled["labels"].value_counts().reset_index()

,labels,count
0,9,2000
1,1,2000
2,8,2000
3,10,2000
4,3,2000
5,11,2000
6,15,1995
7,13,1989
8,12,1988
9,7,1988


In [17]:
length_dist = df_undersampled["post"].str.len().describe()
print(length_dist)

count    31843.000000
mean      1365.636121
std        862.010034
min          2.000000
25%        342.000000
50%       1885.000000
75%       2108.000000
max       2562.000000
Name: post, dtype: float64


In [18]:
df_copy = df_undersampled.loc[df_undersampled["post"].str.len() > 20].reset_index(drop = True)
length_dist = df_copy["post"].str.len().describe()
print(length_dist)

df_copy["labels"].value_counts().reset_index()
print(len(df_undersampled))
print(len(df_copy))

count    31690.000000
mean      1372.165226
std        858.939089
min         21.000000
25%        350.000000
50%       1889.000000
75%       2108.000000
max       2562.000000
Name: post, dtype: float64
31843
31690


In [19]:
short_posts = df_undersampled[df_undersampled["post"].str.len() < 20]
print(f"Anzahl kurzer Posts: {len(short_posts)}")
print("\n--- 30 zufällige Posts unter 50 Zeichen ---\n")
print(short_posts["post"].sample(30, random_state=42).to_string(index=False))

Anzahl kurzer Posts: 138

--- 30 zufällige Posts unter 50 Zeichen ---

title says it all.'
know. .. you ': : '
         neediness.
           <phone>'
   sheer </s> stout
                 x'
              stout
  which ones then?'
  <url> <url> <url>
  welcome. :happy:'
     I'm sorry I...
i don't have a job.
            skunk.'
 😆😆 no! you first!'
      yeah yeah xd'
story of my life!!'
            chose.'
              nap?'
                ok'
            :ball:'
              <url>
    max payne slays
          so true.'
          yes. lol'
     he is a rebel.
 would be slogan! '
               me.'
     tubular dude.'
        james bond'
            <mbti>'


In [ ]:
from datasets import load_dataset
# hf_dict = raw_datasets = load_dataset("DrinkIcedT/mbti")

# df_train = hf_dict["train"].to_pandas()
df_train = pd.read_csv("..\data\mbti_10000\mbti_train.csv")

# Gibt die 10 Zeilen mit den kürzesten Strings zurück
kürzeste_10_train = df_train.loc[df_train['post'].str.len().nsmallest(10).index]
print(kürzeste_10_train[['post']])

# Temporäre Spalte für die Länge erstellen und sortieren
df_sorted_train = df_train.assign(laenge=df_train['post'].str.len()).sort_values('laenge')

# Top 10 anzeigen
print(df_sorted_train.head(10))

# Gibt die 10 Zeilen mit den kürzesten Strings zurück
kürzeste_10 = df.loc[df_copy['post'].str.len().nsmallest(10).index]
print(kürzeste_10[['post']])

# Temporäre Spalte für die Länge erstellen und sortieren
df_sorted = df_copy.assign(laenge=df_copy['post'].str.len()).sort_values('laenge')

# Top 10 anzeigen
print(df_sorted.head(10))

In [ ]:
ergebnis = df_train[df_train['post'].str.contains("I'm sorry I didn't", case=False, na=False)]
print(ergebnis)


In [ ]:
# Annahme: 'posts' ist der Name deiner Text-Spalte
anzahl_kurz = (del_df['post'].str.len() < 100).sum()
del_sorted = del_df.assign(laenge=del_df['post'].str.len()).sort_values('laenge')

print(f"Es gibt {anzahl_kurz} Texte mit weniger als 100 Zeichen.")
print(del_sorted.head(10))

In [ ]:
anzahl_kurz = (bt_df['post'].str.len() < 100).sum()
bt_sorted = bt_df.assign(laenge=bt_df['post'].str.len()).sort_values('laenge')

print(f"Es gibt {anzahl_kurz} Texte mit weniger als 100 Zeichen.")
print(bt_sorted.head(10))

In [ ]:
anzahl_kurz = (syn_df_final['post'].str.len() < 100).sum()
syn_sorted = syn_df_final.assign(laenge=syn_df_final['post'].str.len()).sort_values('laenge')

print(f"Es gibt {anzahl_kurz} Texte mit weniger als 100 Zeichen.")
print(syn_sorted.head(10))

In [ ]:
from datasets import ClassLabel, DatasetDict, Dataset

df_undersampled = df_copy
#df_undersampled = df_undersampled.loc[df_undersampled["post"].str.len() > 85].reset_index(drop = True)

df_undersampled = Dataset.from_pandas(df_undersampled, preserve_index=False).shuffle()


df_undersampled = df_undersampled.class_encode_column("labels")
mbti_labels = ["ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP", 
               "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP"]


new_features = df_undersampled.features.copy()
new_features["labels"] = ClassLabel(names=mbti_labels)

df_undersampled = df_undersampled.cast(new_features)



KeyError: "['author'] not found in axis"

In [26]:
df_undersampled["labels"]

[1,
 11,
 8,
 7,
 2,
 10,
 1,
 11,
 6,
 15,
 13,
 12,
 1,
 8,
 14,
 3,
 8,
 9,
 15,
 11,
 0,
 10,
 10,
 1,
 12,
 0,
 10,
 11,
 9,
 15,
 3,
 6,
 4,
 1,
 8,
 9,
 9,
 1,
 6,
 13,
 1,
 14,
 11,
 11,
 1,
 6,
 5,
 4,
 13,
 4,
 9,
 10,
 11,
 9,
 14,
 4,
 1,
 13,
 0,
 0,
 12,
 0,
 3,
 7,
 13,
 12,
 2,
 10,
 4,
 6,
 14,
 13,
 13,
 8,
 10,
 9,
 10,
 11,
 8,
 15,
 6,
 3,
 1,
 4,
 11,
 12,
 7,
 1,
 4,
 5,
 5,
 13,
 2,
 3,
 8,
 7,
 1,
 6,
 10,
 0,
 0,
 6,
 3,
 12,
 5,
 11,
 8,
 14,
 0,
 10,
 0,
 0,
 10,
 0,
 4,
 0,
 5,
 5,
 1,
 2,
 6,
 4,
 10,
 3,
 6,
 10,
 4,
 0,
 1,
 10,
 7,
 1,
 6,
 8,
 7,
 4,
 1,
 1,
 10,
 10,
 8,
 10,
 7,
 2,
 13,
 11,
 10,
 11,
 7,
 7,
 8,
 3,
 4,
 14,
 5,
 11,
 2,
 2,
 11,
 3,
 7,
 9,
 9,
 9,
 3,
 15,
 10,
 6,
 13,
 2,
 3,
 4,
 15,
 13,
 15,
 6,
 9,
 15,
 1,
 8,
 10,
 6,
 1,
 1,
 7,
 7,
 13,
 5,
 4,
 6,
 15,
 15,
 8,
 5,
 0,
 15,
 1,
 1,
 3,
 15,
 5,
 12,
 11,
 13,
 13,
 1,
 11,
 2,
 8,
 13,
 13,
 12,
 15,
 4,
 1,
 5,
 8,
 7,
 0,
 15,
 2,
 15,
 10,
 15,
 5,
 5,
 5,
 12,
 2,


In [23]:
df_undersampled_filtered = df_undersampled.loc[df_undersampled["post"].str.len() > 85].reset_index(drop = True)
anzahl_kurz = (df_undersampled_filtered['post'].str.len() < 100).sum()
sorted = df_undersampled_filtered.assign(laenge=syn_df_final['post'].str.len()).sort_values('laenge')

print(f"Es gibt {anzahl_kurz} Texte mit weniger als 100 Zeichen.")
print(sorted.head(10))

df_undersampled_filtered["label"].value_counts().reset_index()

AttributeError: 'Dataset' object has no attribute 'loc'

In [33]:
from datasets import load_from_disk
#hf_dict = load_from_disk("C:/Users/Tim/src/MA/data/mbti_hfdict")
#hf_dict["train"] = df_undersampled

#df_hfdict = df_hfdict.remove_columns("author")
df_hfdict["train"] = df_undersampled

df_hfdict["validation"] = df_hfdict["validation"].shuffle()
df_hfdict["test"] = df_hfdict["test"].shuffle()

df_hfdict.save_to_disk("..\data\mbti_agg_balanced")
df_hfdict.push_to_hub("DrinkIcedT/mbti_agg_balanced")

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_2692\794597157.py:11: SyntaxWarning: invalid escape sequence '\d'
  df_hfdict.save_to_disk("..\data\mbti_agg_balanced")


Saving the dataset (0/1 shards):   0%|          | 0/31690 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5460 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5396 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/DrinkIcedT/mbti_agg_balanced/commit/21124ed1f5ae433a9c985a9f5fd7cf7cc538361a', commit_message='Upload dataset', commit_description='', oid='21124ed1f5ae433a9c985a9f5fd7cf7cc538361a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DrinkIcedT/mbti_agg_balanced', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DrinkIcedT/mbti_agg_balanced'), pr_revision=None, pr_num=None)

In [32]:
df_hfdict["validation"] = df_hfdict["validation"].shuffle()